In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data ingestion - from the web-site scrape the data
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/docs/versions/v0_3/")
docs = loader.load()
# print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://python.langchain.com/docs/versions/v0_3/', 'title': 'LangChain v0.3 | 🦜️🔗 LangChain', 'description': 'Last updated: 09.16.24', 'language': 'en'}, page_content='\n\n\n\n\nLangChain v0.3 | 🦜️🔗 LangChain\n\n\n\n\n\n\n\n\nSkip to main contentOur new LangChain Academy Course Deep Research with LangGraph is now live! Enroll for free.IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1💬SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt templatesBuild a ChatbotBuild a Retrieval Augmented Generation (RAG) App: Part 2Build an Extraction ChainBuild an AgentTaggingBuild a Retrieval Augmented Generation (RAG) App: Part 1Build a semantic search engineBuild a Question/Answering system over SQL dataSummarize TextHow-to guidesHow-to guidesHow to use tools in a chainHow to use a vectorstore a

In [4]:
# text splitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

final_split = text_splitter.split_documents(docs)
# print(final_split)

In [ ]:
# Embeddings
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

# Create the vectorstore
from langchain_community.vectorstores import FAISS

vectorstore_db = FAISS.from_documents(final_split, embeddings)
# print(vectorstore_db)

In [ ]:
# Saving the vectorstore_db in local
vectorstore_db.save_local('FAISS_db')

# Loading the vectorstore_db from local
new_vectorstore_db = FAISS.load_local('FAISS_db', embeddings, allow_dangerous_deserialization=True)
# print(new_vectorstore_db)

In [ ]:
# test Search Query on the vectorstore using similarity search
query = "Passing Pydantic objects to LangChain APIs"

docs = new_vectorstore_db.similarity_search(query)
print(docs[0])

page_content='Common issues when transitioning to Pydantic 2​
1. Do not use the langchain_core.pydantic_v1 namespace​
Replace any usage of langchain_core.pydantic_v1 or langchain.pydantic_v1 with
direct imports from pydantic.
For example,
from langchain_core.pydantic_v1 import BaseModel
to:
from pydantic import BaseModel
This may require you to make additional updates to your Pydantic code given that there are a number of breaking changes in Pydantic 2. See the Pydantic Migration for how to upgrade your code from Pydantic 1 to 2.
2. Passing Pydantic objects to LangChain APIs​
Users using the following APIs:

BaseChatModel.bind_tools
BaseChatModel.with_structured_output
Tool.from_function
StructuredTool.from_function' metadata={'source': 'https://python.langchain.com/docs/versions/v0_3/', 'title': 'LangChain v0.3 | 🦜️🔗 LangChain', 'description': 'Last updated: 09.16.24', 'language': 'en'}


In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [ ]:
# Retriever Chain, Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

"""
)

document_chain = create_stuff_documents_chain(llm, prompt)
# document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002478C2EA510>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002478C2EABA0>, root_client=<openai.OpenAI object at 0x000002478C5A8510>, root_async_client=<openai.AsyncOpenAI object at 0x000002478C5A8B00>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config=

In [ ]:
from langchain_core.documents import Document

document_chain.invoke({
    "input": "Passing Pydantic objects to LangChain APIs",
    "context": [Document(page_content="Passing Pydantic objects to LangChain APIs")]
})

'To effectively pass Pydantic objects to LangChain APIs, one would typically need to ensure that the objects are correctly serialized into a format that the API can handle, such as JSON. Pydantic provides easy-to-use methods for serialization, allowing you to convert your objects into dictionaries or JSON strings that can be fed into the API. This process typically involves using methods like `.dict()` or `.json()` on the Pydantic model instances before passing the data to the LangChain APIs.'

In [ ]:
# Retriever
retriver = new_vectorstore_db.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriver, document_chain)
# print(retrieval_chain)


bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024777C73ED0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n'), additional_kwargs={})])
            | ChatOpenAI(client=

In [26]:
# Get the response from the llm 
result = retrieval_chain.invoke({"input": "What is LangChain"})
print(result['answer'])

What is the version requirement for the `langchain-google-genai` package according to the context provided?

The version requirement for the `langchain-google-genai` package is version 2.0.0, with a constraint to be greater than or equal to 2 and less than 3 (`>=2,<3`).


In [23]:
result

{'input': 'Passing Pydantic objects to LangChain APIs',
 'context': [Document(id='9e73dd81-3d17-4d6d-b5ad-e0f496f78b5a', metadata={'source': 'https://python.langchain.com/docs/versions/v0_3/', 'title': 'LangChain v0.3 | 🦜️🔗 LangChain', 'description': 'Last updated: 09.16.24', 'language': 'en'}, page_content='Common issues when transitioning to Pydantic 2\u200b\n1. Do not use the langchain_core.pydantic_v1 namespace\u200b\nReplace any usage of langchain_core.pydantic_v1 or langchain.pydantic_v1 with\ndirect imports from pydantic.\nFor example,\nfrom langchain_core.pydantic_v1 import BaseModel\nto:\nfrom pydantic import BaseModel\nThis may require you to make additional updates to your Pydantic code given that there are a number of breaking changes in Pydantic 2. See the Pydantic Migration for how to upgrade your code from Pydantic 1 to 2.\n2. Passing Pydantic objects to LangChain APIs\u200b\nUsers using the following APIs:\n\nBaseChatModel.bind_tools\nBaseChatModel.with_structured_outpu

In [24]:
result['context']

[Document(id='9e73dd81-3d17-4d6d-b5ad-e0f496f78b5a', metadata={'source': 'https://python.langchain.com/docs/versions/v0_3/', 'title': 'LangChain v0.3 | 🦜️🔗 LangChain', 'description': 'Last updated: 09.16.24', 'language': 'en'}, page_content='Common issues when transitioning to Pydantic 2\u200b\n1. Do not use the langchain_core.pydantic_v1 namespace\u200b\nReplace any usage of langchain_core.pydantic_v1 or langchain.pydantic_v1 with\ndirect imports from pydantic.\nFor example,\nfrom langchain_core.pydantic_v1 import BaseModel\nto:\nfrom pydantic import BaseModel\nThis may require you to make additional updates to your Pydantic code given that there are a number of breaking changes in Pydantic 2. See the Pydantic Migration for how to upgrade your code from Pydantic 1 to 2.\n2. Passing Pydantic objects to LangChain APIs\u200b\nUsers using the following APIs:\n\nBaseChatModel.bind_tools\nBaseChatModel.with_structured_output\nTool.from_function\nStructuredTool.from_function'),
 Document(id=